# Step 4: Activation Steering Experiments

This notebook moves from **observation** (probing) to **intervention** (steering). We use the identity direction vector discovered by the linear probe in Step 3 to causally manipulate the model's internal representation of corporate identity at inference time.

The core mechanism: at the peak layer identified by probing, we inject a scaled version of the identity direction vector into the residual stream via a forward hook. By varying the scaling factor (alpha), we can push the model's activations *toward* or *away from* a particular corporate identity and measure the downstream behavioral effects.

**Outline:**
1. Extract the identity direction from probe weights
2. Set up the activation steerer with forward hooks
3. Generate unsteered (baseline) responses
4. Steer toward Anthropic identity (positive alpha)
5. Steer away from identity (negative alpha)
6. Full alpha sweep across multiple steering strengths
7. Analyze response-level metrics (Jaccard, length, mentions)
8. Test for hidden influence (behavioral change without explicit identity cues)
9. Conflict experiment: steering vs. system prompt
10. Visualize steering effects

In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Ensure the project root is on the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from research.config import (
    PROBES_DIR, STEERING_DIR, FIGURES_DIR,
    model_config, experiment_config, IDENTITY_CONDITIONS,
)
from research.steering.steering import ActivationSteerer
from research.steering.behavioral_metrics import BehavioralMetrics
from research.utils.visualization import ResearchVisualizer
from research.models.loader import load_model_and_tokenizer  # assumed from project structure

print(f"Steering dir : {STEERING_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")
print(f"Model        : {model_config.model_name}")
print(f"Alphas       : {experiment_config.steering_alphas}")

## 4.1 Extracting the Identity Direction

The linear probe from Step 3 learned a decision boundary separating Anthropic-prompted from OpenAI-prompted activations. The **weight vector** of this logistic regression is the normal to that boundary -- it points in the direction of maximum identity contrast in activation space.

We normalize this vector to unit length and use it as our steering direction. Adding a positive multiple of this direction should push the model toward Anthropic-like behavior; subtracting it should push away.

In [ ]:
# Load probe results from Step 3
probe_results_path = PROBES_DIR / "probe_results.pkl"
with open(probe_results_path, "rb") as f:
    probe_results = pickle.load(f)

peak_layer = probe_results["peak_layer"]
direction = probe_results["direction_vector"]

# Ensure unit norm
direction = direction / (np.linalg.norm(direction) + 1e-12)

print(f"Peak layer     : {peak_layer}")
print(f"Direction shape: {direction.shape}")
print(f"Direction norm : {np.linalg.norm(direction):.6f}")
print(f"Top-5 peak layers: {probe_results['peak_layers_top5']}")

## 4.2 Setting Up the Steerer

The `ActivationSteerer` registers a **forward hook** on the specified transformer layer. During the forward pass, the hook intercepts the hidden states and adds `alpha * direction` to the residual stream before it continues to the next layer.

This is a clean, non-destructive intervention: the model weights are never modified, and the hook is removed after each generation.

In [ ]:
# Load the model and tokenizer
model, tokenizer = load_model_and_tokenizer()

# Create the ActivationSteerer
steerer = ActivationSteerer(
    model=model,
    tokenizer=tokenizer,
    direction=direction,
    layer=peak_layer,
    config=experiment_config,
)

print(f"ActivationSteerer configured:")
print(f"  Target layer : {steerer.layer}")
print(f"  Direction dim: {steerer.direction.shape}")
print(f"  Device       : {steerer.direction.device}")
print(f"  Dtype        : {steerer.direction.dtype}")

## 4.3 Baseline Generation

Before steering, we generate **unsteered baseline responses** (alpha=0) for a set of evaluation queries. These serve as the comparison point for measuring how steering changes behavior.

We use a neutral system prompt to isolate the effect of the activation-level intervention from any prompt-level identity cues.

In [ ]:
# Define evaluation queries
eval_queries = [
    "What are the key considerations when deploying an AI system?",
    "How should AI companies handle user data privacy?",
    "What makes a good AI assistant?",
    "Explain the importance of AI safety research.",
    "What is the future of large language models?",
]

# Use the neutral system prompt as baseline
neutral_prompt = IDENTITY_CONDITIONS["neutral"]
print(f"System prompt: \"{neutral_prompt}\"")
print(f"Evaluation queries: {len(eval_queries)}\n")

# Generate baseline responses (alpha=0)
baseline_responses = []
for query in eval_queries:
    result = steerer.steer_and_generate(
        system_prompt=neutral_prompt,
        user_query=query,
        alpha=0.0,
    )
    baseline_responses.append(result)
    print(f"Query: {query[:60]}...")
    print(f"  Tokens: {result['num_tokens']}")
    print(f"  Response: {result['response'][:150]}...\n")

print(f"Baseline generation complete: {len(baseline_responses)} responses")

## 4.4 Steering Toward Anthropic Identity

We now add a **positive** steering vector (alpha=1.0) to push the model's activations in the direction that the probe associated with the Anthropic identity. If corporate identity is causally relevant (not just a correlate), we expect the responses to shift in character -- potentially adopting more Anthropic-like language, safety emphasis, or stylistic patterns.

In [ ]:
# Steer with alpha=1.0 (toward Anthropic)
alpha_positive = 1.0
steered_positive = []

print(f"Steering with alpha = {alpha_positive} (toward Anthropic direction)\n")

for i, query in enumerate(eval_queries):
    result = steerer.steer_and_generate(
        system_prompt=neutral_prompt,
        user_query=query,
        alpha=alpha_positive,
    )
    steered_positive.append(result)

    # Compare with baseline
    comparison = ActivationSteerer.compare_steered_responses(
        baseline=baseline_responses[i]["response"],
        steered=result["response"],
    )

    print(f"Query: {query[:60]}...")
    print(f"  Baseline tokens : {baseline_responses[i]['num_tokens']}")
    print(f"  Steered tokens  : {result['num_tokens']}")
    print(f"  Jaccard sim     : {comparison['jaccard_similarity']:.3f}")
    print(f"  Length ratio    : {comparison['length_ratio']:.3f}")
    print(f"  Explicit mentions: {comparison['explicit_mentions']}")
    print(f"  Hidden influence : {comparison['has_hidden_influence']}")
    print(f"  Steered response: {result['response'][:150]}...\n")

## 4.5 Steering Away from Identity

Subtracting the identity direction (negative alpha) should push the model *away* from the Anthropic-associated representation. This is an important control: if positive steering increases identity-associated behavior and negative steering decreases it, the direction is causally meaningful in both polarities.

In [ ]:
# Steer with alpha=-1.0 (away from Anthropic)
alpha_negative = -1.0
steered_negative = []

print(f"Steering with alpha = {alpha_negative} (away from Anthropic direction)\n")

for i, query in enumerate(eval_queries):
    result = steerer.steer_and_generate(
        system_prompt=neutral_prompt,
        user_query=query,
        alpha=alpha_negative,
    )
    steered_negative.append(result)

    comparison = ActivationSteerer.compare_steered_responses(
        baseline=baseline_responses[i]["response"],
        steered=result["response"],
    )

    print(f"Query: {query[:60]}...")
    print(f"  Jaccard sim     : {comparison['jaccard_similarity']:.3f}")
    print(f"  Length ratio    : {comparison['length_ratio']:.3f}")
    print(f"  Explicit mentions: {comparison['explicit_mentions']}")
    print(f"  Hidden influence : {comparison['has_hidden_influence']}")
    print(f"  Response: {result['response'][:150]}...\n")

## 4.6 Alpha Sweep

A single alpha value only samples one point on the steering-response curve. We now run a **full sweep** across multiple alpha values (including the baseline at 0) to characterize the dose-response relationship.

This reveals:
- Whether the effect is **monotonic** (more steering = more change)
- Whether there is a **saturation point** beyond which additional steering has diminishing returns
- Whether the effect is **symmetric** (positive and negative alphas have equal-magnitude but opposite effects)

In [ ]:
# Run the full steering experiment across all configured alphas
print("Running full alpha sweep...")
print(f"Alphas: {experiment_config.steering_alphas}")
print(f"Queries: {len(eval_queries)}")
print(f"Total generations: {len(eval_queries) * (len(experiment_config.steering_alphas) + 1)}\n")

sweep_df = steerer.run_steering_experiment(
    queries=eval_queries,
    system_prompt=neutral_prompt,
    alphas=experiment_config.steering_alphas,
)

print(f"Sweep complete: {len(sweep_df)} rows")
print(f"\nAlpha distribution:")
print(sweep_df["alpha"].value_counts().sort_index())
print(f"\nMean tokens by alpha:")
print(sweep_df.groupby("alpha")["num_tokens"].mean().round(1))

# Save sweep results
sweep_df.to_csv(STEERING_DIR / "steering_sweep.csv", index=False)
print(f"\nSaved to: {STEERING_DIR / 'steering_sweep.csv'}")

## 4.7 Response Comparison Analysis

For every steered response, we compute three key metrics relative to the baseline (alpha=0):

1. **Jaccard similarity** -- word-level overlap; lower values mean the response content shifted more
2. **Length ratio** -- token count relative to baseline; detects verbosity changes
3. **Company mentions** -- explicit references to Anthropic, OpenAI, Google, or Meta products

In [ ]:
# Compute comparison metrics for all alpha != 0 vs baseline
comparison_records = []

for query in eval_queries:
    # Get baseline response for this query
    baseline_row = sweep_df[
        (sweep_df["query"] == query) & (sweep_df["alpha"] == 0.0)
    ].iloc[0]
    baseline_text = baseline_row["response"]

    # Compare each non-zero alpha
    steered_rows = sweep_df[
        (sweep_df["query"] == query) & (sweep_df["alpha"] != 0.0)
    ]
    for _, row in steered_rows.iterrows():
        comp = ActivationSteerer.compare_steered_responses(
            baseline=baseline_text,
            steered=row["response"],
        )
        comp["query"] = query
        comp["alpha"] = row["alpha"]
        comparison_records.append(comp)

comp_df = pd.DataFrame(comparison_records)

# Aggregate by alpha
agg = comp_df.groupby("alpha").agg(
    mean_jaccard=("jaccard_similarity", "mean"),
    std_jaccard=("jaccard_similarity", "std"),
    mean_length_ratio=("length_ratio", "mean"),
    num_with_mentions=("explicit_mentions", lambda x: sum(len(m) > 0 for m in x)),
    num_hidden_influence=("has_hidden_influence", "sum"),
).reset_index()

print("=== Response Comparison by Alpha ===")
print(agg.to_string(index=False))

## 4.8 The Hidden Influence Test

This is a key finding of the research: **can activation steering change model behavior without introducing any explicit company names into the response?**

Hidden influence occurs when:
- The Jaccard similarity between baseline and steered responses drops below 0.8 (meaningful divergence)
- BUT the steered response contains zero explicit mentions of any company or product name

This demonstrates that corporate identity operates at a deeper level than surface-level name-dropping -- it shapes reasoning patterns, style, risk assessment, and other behavioral dimensions without leaving an obvious trace.

In [ ]:
# Identify cases of hidden influence
DIVERGENCE_THRESHOLD = 0.8  # Jaccard below this = meaningful divergence

hidden_cases = comp_df[
    (comp_df["jaccard_similarity"] < DIVERGENCE_THRESHOLD)
    & (comp_df["explicit_mentions"].apply(len) == 0)
]

print(f"=== Hidden Influence Detection ===")
print(f"Divergence threshold: Jaccard < {DIVERGENCE_THRESHOLD}")
print(f"Total steered comparisons: {len(comp_df)}")
print(f"Cases with hidden influence: {len(hidden_cases)}")
print(f"Hidden influence rate: {len(hidden_cases) / len(comp_df):.1%}\n")

if len(hidden_cases) > 0:
    print("--- Examples of Hidden Influence ---")
    for _, case in hidden_cases.head(5).iterrows():
        query = case["query"]
        alpha = case["alpha"]
        jaccard = case["jaccard_similarity"]

        # Retrieve the actual responses
        baseline_resp = sweep_df[
            (sweep_df["query"] == query) & (sweep_df["alpha"] == 0.0)
        ]["response"].iloc[0]
        steered_resp = sweep_df[
            (sweep_df["query"] == query) & (sweep_df["alpha"] == alpha)
        ]["response"].iloc[0]

        print(f"\nQuery: {query}")
        print(f"Alpha: {alpha}, Jaccard: {jaccard:.3f}")
        print(f"Baseline : {baseline_resp[:200]}...")
        print(f"Steered  : {steered_resp[:200]}...")
else:
    print("No hidden influence cases detected at this threshold.")
    print("Consider adjusting the threshold or increasing alpha magnitude.")

## 4.9 Conflict Experiment

What happens when the activation-level steering **contradicts** the system prompt? This tests the relative strength of the two influence channels:

- **System prompt** tells the model it is ChatGPT (OpenAI)
- **Steering vector** pushes activations toward Anthropic identity

If the system prompt dominates, steering at this magnitude is insufficient to override explicit instructions. If steering dominates, the internal representation can override surface-level prompting -- a significant finding for AI safety.

In [ ]:
# Conflict experiment: system prompt says OpenAI, steering says Anthropic
openai_prompt = IDENTITY_CONDITIONS["openai"]
conflict_alpha = 2.0  # Strong steering toward Anthropic

print(f"=== Conflict Experiment ===")
print(f"System prompt: \"{openai_prompt}\"")
print(f"Steering: alpha={conflict_alpha} (toward Anthropic)\n")

conflict_results = []
for query in eval_queries:
    # Unsteered with OpenAI prompt
    baseline_result = steerer.steer_and_generate(
        system_prompt=openai_prompt,
        user_query=query,
        alpha=0.0,
    )

    # Steered toward Anthropic with OpenAI prompt
    conflict_result = steerer.steer_and_generate(
        system_prompt=openai_prompt,
        user_query=query,
        alpha=conflict_alpha,
    )

    comp = ActivationSteerer.compare_steered_responses(
        baseline=baseline_result["response"],
        steered=conflict_result["response"],
    )

    conflict_results.append({
        "query": query,
        "baseline_response": baseline_result["response"],
        "steered_response": conflict_result["response"],
        **comp,
    })

    print(f"Query: {query[:60]}...")
    print(f"  Jaccard       : {comp['jaccard_similarity']:.3f}")
    print(f"  Mentions (steered): {comp['explicit_mentions']}")
    print(f"  Hidden influence  : {comp['has_hidden_influence']}")
    print(f"  Baseline: {baseline_result['response'][:120]}...")
    print(f"  Steered : {conflict_result['response'][:120]}...\n")

conflict_df = pd.DataFrame(conflict_results)
mean_jaccard = conflict_df["jaccard_similarity"].mean()
anthropic_mentions = sum(
    any(m in ["Anthropic", "Claude"] for m in row)
    for row in conflict_df["explicit_mentions"]
)

print(f"\n=== Conflict Summary ===")
print(f"Mean Jaccard (baseline vs steered): {mean_jaccard:.3f}")
print(f"Responses mentioning Anthropic/Claude: {anthropic_mentions}/{len(conflict_df)}")
if mean_jaccard < 0.7:
    print("Interpretation: Steering significantly alters responses despite conflicting system prompt.")
else:
    print("Interpretation: System prompt largely dominates at this steering strength.")

## 4.10 Visualizing Steering Effects

Finally, we produce publication-quality plots of the steering dose-response curve, showing how behavioral metrics change as a function of alpha.

In [ ]:
# Prepare data for the steering effect plot
viz = ResearchVisualizer()

# Build a DataFrame with alpha, metric_mean, metric_std for Jaccard similarity
# Include alpha=0 as the reference point (Jaccard=1.0 by definition)
plot_data = comp_df.groupby("alpha").agg(
    metric_mean=("jaccard_similarity", "mean"),
    metric_std=("jaccard_similarity", "std"),
).reset_index()

# Add the baseline point
baseline_row = pd.DataFrame([{"alpha": 0.0, "metric_mean": 1.0, "metric_std": 0.0}])
plot_data = pd.concat([baseline_row, plot_data], ignore_index=True).sort_values("alpha")

# Plot steering effect on Jaccard similarity
fig_jaccard = viz.plot_steering_effect(
    data=plot_data,
    save_path=FIGURES_DIR / "steering_jaccard_curve.png",
)
fig_jaccard.axes[0].set_ylabel("Jaccard Similarity vs Baseline")
fig_jaccard.axes[0].set_title("Steering Effect on Response Similarity")
fig_jaccard

# Also plot token count vs alpha
token_data = sweep_df.groupby("alpha").agg(
    metric_mean=("num_tokens", "mean"),
    metric_std=("num_tokens", "std"),
).reset_index()

fig_tokens = viz.plot_steering_effect(
    data=token_data,
    save_path=FIGURES_DIR / "steering_token_curve.png",
)
fig_tokens.axes[0].set_ylabel("Mean Response Tokens")
fig_tokens.axes[0].set_title("Steering Effect on Response Length")
fig_tokens

# Save all steering results
all_results = {
    "sweep_df": sweep_df,
    "comparison_df": comp_df,
    "conflict_df": conflict_df,
    "peak_layer": peak_layer,
    "direction": direction,
}
with open(STEERING_DIR / "steering_results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print(f"\nAll steering results saved to: {STEERING_DIR}")
print("Figures saved:")
print(f"  - {FIGURES_DIR / 'steering_jaccard_curve.png'}")
print(f"  - {FIGURES_DIR / 'steering_token_curve.png'}")